In [ ]:
#Cloning github repository
! git clone https://github.com/ahmed-roushdy16004392/data-engineering-project-1.git

Cloning into 'data-engineering-project-1'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 31 (delta 8), reused 5 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 1.42 MiB | 8.81 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [ ]:
#Checking files inside repository
import os

os.listdir('/content/data-engineering-project-1')

['app.py',
 '.gitignore',
 'README.md',
 'Notebook',
 '.git',
 'requirements.txt',
 'Data']

In [ ]:
#Checking files inside Data
os.listdir('/content/data-engineering-project-1/Data')

['panel_borough_year.csv',
 'Raw Data',
 'housing_clean.csv',
 'crashes_clean.csv',
 'council_districts.geojson',
 'panel_district_year.csv']

In [ ]:
#Importing needed libraries and setting up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", palette="Set2")

# Load the datasets
base_path = "/content/data-engineering-project-1/Data/"

panel_b = pd.read_csv(base_path + "panel_borough_year.csv")
c_web = pd.read_csv(base_path + "crashes_clean.csv")

# Inspect column names
print("BOROUGH PANEL COLUMNS:")
print(panel_b.columns.tolist())

print("\nCRASH DATA COLUMNS:")
print(c_web.columns.tolist())

BOROUGH PANEL COLUMNS:
['borough_clean', 'year', 'crashes', 'fatal_crashes', 'fatalities', 'severe_injuries', 'building_records', 'affordable_units', 'new_units', 'preserved_units']

CRASH DATA COLUMNS:
['Collision ID', 'date_clean', 'year', 'weekday', 'hour', 'borough_clean', 'council_final', 'Latitude', 'Longitude', 'On Street', 'severity', 'Total Severe Injuries', 'Total Non-Severe Injuries', 'Total Fatalities', 'mass_casualty']


In [ ]:
# Create a copy of the borough panel
housing_chart = panel_b[
    ['borough_clean', 'year', 'new_units', 'preserved_units']
].copy()

# Convert numerical columns to numeric format
housing_chart['year'] = pd.to_numeric(
    housing_chart['year'], errors='coerce'
)

housing_chart['new_units'] = pd.to_numeric(
    housing_chart['new_units'], errors='coerce'
)

housing_chart['preserved_units'] = pd.to_numeric(
    housing_chart['preserved_units'], errors='coerce'
)

# Remove rows without borough or year
housing_chart = housing_chart.dropna(
    subset=['borough_clean', 'year']
)

# Check for duplicate borough-year combinations
print("Duplicate borough-year records:",
      housing_chart.duplicated(['borough_clean', 'year']).sum())

# Aggregate in case duplicate borough-year records exist
housing_chart = (
    housing_chart.groupby(
        ['borough_clean', 'year'],
        as_index=False
    )[['new_units', 'preserved_units']]
    .sum(min_count=1)
)

display(housing_chart)

Duplicate borough-year records: 0


,borough_clean,year,new_units,preserved_units
0,Bronx,2022,2800,9206
1,Bronx,2023,5240,12182
2,Bronx,2024,4388,4985
3,Bronx,2025,5089,1954
4,Brooklyn,2022,3056,413
5,Brooklyn,2023,4199,2847
6,Brooklyn,2024,3650,1329
7,Brooklyn,2025,5477,546
8,Manhattan,2022,1242,1454
9,Manhattan,2023,950,2752


In [ ]:
# Reshape data for stacked bar chart
housing_long = housing_chart.melt(
    id_vars=['borough_clean', 'year'],
    value_vars=['new_units', 'preserved_units'],
    var_name='Housing Type',
    value_name='Affordable Units'
)

# Make category names more readable
housing_long['Housing Type'] = housing_long['Housing Type'].replace({
    'new_units': 'New Units',
    'preserved_units': 'Preserved Units'
})

# Create chart
fig_q2 = px.bar(
    housing_long,
    x='borough_clean',
    y='Affordable Units',
    color='Housing Type',
    facet_col='year',
    facet_col_wrap=2,
    barmode='stack',
    title='Affordable Housing Units Completed by Borough (2022–2025)',
    labels={
        'borough_clean': 'Borough',
        'Affordable Units': 'Number of Affordable Units',
        'Housing Type': 'Housing Category',
        'year': 'Year'
    },
    color_discrete_map={
        'New Units': '#2878B5',
        'Preserved Units': '#F4A261'
    }
)

fig_q2.update_layout(
    template='plotly_white',
    height=750,
    width=1100,
    legend_title='Housing Type',
    title_x=0.5
)

fig_q2.update_xaxes(tickangle=-30)

fig_q2.show()

**What the chart shows:**

The chart compares the number of affordable housing units completed across NYC boroughs between 2022 and 2025, distinguishing between newly constructed and preserved units. It highlights differences in the distribution of housing completions across boroughs and how the relative contribution of new construction and preservation varies over time.

**Limits:**

The chart presents absolute unit counts rather than rates relative to population or existing housing stock, and therefore does not account for differences in borough size.

In [ ]:
print("Severity categories:")
print(c_web['severity'].value_counts(dropna=False))

print("\nMissing crash dates:")
print(c_web['date_clean'].isna().sum())

print("\nDate range:")
print(c_web['date_clean'].min())
print(c_web['date_clean'].max())

Severity categories:
severity
Severe injury    11033
Fatal              925
Name: count, dtype: int64

Missing crash dates:
0

Date range:
2022-01-01
2025-12-30


In [ ]:
crash_chart = c_web[
    ['Collision ID', 'date_clean', 'severity']
].copy()

# Convert dates
crash_chart['date_clean'] = pd.to_datetime(
    crash_chart['date_clean'],
    errors='coerce'
)

# Standardize severity labels
crash_chart['severity'] = (
    crash_chart['severity']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep 2022–2025
crash_chart = crash_chart[
    crash_chart['date_clean'].dt.year.between(2022, 2025)
].copy()

# Keep only the two serious crash categories
crash_chart = crash_chart[
    crash_chart['severity'].isin([
        'severe injury',
        'fatal'
    ])
].copy()

# Check duplicate collision IDs
print(
    "Duplicate collision IDs:",
    crash_chart['Collision ID'].duplicated().sum()
)

# Extract month
crash_chart['Month'] = (
    crash_chart['date_clean'].dt.month
)

# Standardize display names
crash_chart['Severity'] = crash_chart['severity'].map({
    'severe injury': 'Severe Injury',
    'fatal': 'Fatal'
})

print(crash_chart['Severity'].value_counts())

Duplicate collision IDs: 0
Severity
Severe Injury    11033
Fatal              925
Name: count, dtype: int64


In [ ]:
# Creating line chart
monthly_crashes = (
    crash_chart.groupby(['Month', 'Severity'])
    ['Collision ID']
    .nunique()
    .reset_index(name='Crash Count')
)

# Ensure every month and severity combination exists
all_combinations = pd.MultiIndex.from_product(
    [
        range(1, 13),
        ['Severe Injury', 'Fatal']
    ],
    names=['Month', 'Severity']
)

monthly_crashes = (
    monthly_crashes
    .set_index(['Month', 'Severity'])
    .reindex(all_combinations, fill_value=0)
    .reset_index()
)

print(monthly_crashes)

    Month       Severity  Crash Count
0       1  Severe Injury          658
1       1          Fatal           65
2       2  Severe Injury          671
3       2          Fatal           70
4       3  Severe Injury          813
5       3          Fatal           61
6       4  Severe Injury          735
7       4          Fatal           58
8       5  Severe Injury         1077
9       5          Fatal           79
10      6  Severe Injury         1014
11      6          Fatal           82
12      7  Severe Injury         1134
13      7          Fatal           95
14      8  Severe Injury         1066
15      8          Fatal           98
16      9  Severe Injury         1087
17      9          Fatal           72
18     10  Severe Injury         1042
19     10          Fatal           94
20     11  Severe Injury          900
21     11          Fatal           80
22     12  Severe Injury          836
23     12          Fatal           71


In [ ]:

crash_chart['Year'] = (
    crash_chart['date_clean'].dt.year
)

yearly_monthly_crashes = (
    crash_chart.groupby(
        ['Year', 'Month', 'Severity']
    )['Collision ID']
    .nunique()
    .reset_index(name='Crash Count')
)

fig = px.line(
    yearly_monthly_crashes,
    x='Month',
    y='Crash Count',
    color='Year',
    facet_row='Severity',
    markers=True,
    title='Monthly Serious Crash Trends by Year and Severity',
    labels={
        'Month': 'Month',
        'Crash Count': 'Number of Crashes',
        'Year': 'Year'
    }
)

fig.update_layout(
    template='plotly_white',
    height=700,
    title_x=0.5
)

fig.show()

**What the chart shows:**

The chart suggests a seasonal pattern in severe-injury crashes, with counts generally increasing from the beginning of the year and reaching higher levels between May and September, followed by a decline toward the end of the year. Fatal crashes show considerably lower monthly counts and more irregular fluctuations, without a consistent peak month across 2022–2025. Overall, severe-injury crashes appear to follow a more noticeable mid-year pattern than fatal crashes.

**Limits:**

These figures represent monthly crash counts rather than rates adjusted for traffic volume or exposure, so they show associations and do not establish why crashes increase or decrease during particular months.